# CyberShield: train on Colab with extra datasets

1. Upload the project zip (Step 1).
2. Add datasets in Step 3. Each dataset becomes `ml/extra/<name>.csv`.
3. Run training (Step 4). Download `backend/model/` (Step 5) and copy it into your local project's `backend/model/`.

Colab's scikit-learn version must match your local one for the `.joblib` to load. Step 2 pins the version this project was verified with; Step 5 prints it.

## Step 1: upload the project zip

In [ ]:
from google.colab import files
import zipfile, os, glob
uploaded = files.upload()          # choose the project zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/proj')
train_py = glob.glob('/content/proj/**/ml/train_model.py', recursive=True)[0]
ROOT = os.path.dirname(os.path.dirname(train_py))
%cd {ROOT}
!ls

## Step 2: install pinned dependencies

In [ ]:
!pip -q install "scikit-learn==1.9.1" "pandas>=2,<3" "joblib>=1.3,<2" "Flask>=3,<4"
import sklearn; print('scikit-learn', sklearn.__version__)

## Step 3: add datasets

`add_dataset` converts any dataframe to the project format and saves it in `ml/extra/`:
- `text_col`: column holding the text
- `is_toxic`: function that takes a row and returns True (toxic) or False (non-toxic)
- `max_per_class`: cap rows per class so one big dataset does not swamp the others (set `None` to keep all)

Check each dataset's license. Send me the dataset links and I'll fill in the exact column mapping for each one.

In [ ]:
import pandas as pd, os
os.makedirs('ml/extra', exist_ok=True)

def add_dataset(name, df, text_col, is_toxic, max_per_class=20000, seed=42):
    out = pd.DataFrame({'text': df[text_col].fillna('').astype(str),
                        'label': df.apply(lambda r: 1 if is_toxic(r) else 0, axis=1)})
    out = out[out.text.str.strip().ne('')]
    if max_per_class:
        out = out.groupby('label', group_keys=False).apply(lambda g: g.sample(min(len(g), max_per_class), random_state=seed))
    out.to_csv(f'ml/extra/{name}.csv', index=False)
    print(name, len(out), out.label.value_counts().to_dict())

In [ ]:
# ---- TEMPLATES: uncomment and edit once you have the dataset ----

# (a) A CSV you uploaded to Colab (e.g. Jigsaw train.csv from Kaggle, which has a 'comment_text' column and a 'toxic' 0/1 column)
# df = pd.read_csv('/content/train.csv')
# add_dataset('jigsaw', df, 'comment_text', lambda r: r['toxic'] >= 1)

# (b) A Hugging Face dataset (replace the placeholder with the dataset id from the link)
# from datasets import load_dataset
# ds = load_dataset('<HF-DATASET-ID>', split='train').to_pandas()
# add_dataset('hinglish', ds, '<TEXT-COLUMN>', lambda r: r['<LABEL-COLUMN>'] == 1)

# (c) A CSV from a URL
# df = pd.read_csv('<RAW-CSV-URL>')
# add_dataset('mydata', df, '<TEXT-COLUMN>', lambda r: r['<LABEL-COLUMN>'] == 1)

## Step 4: train (about 1 to 10 minutes depending on data size)

In [ ]:
!python ml/train_model.py

In [ ]:
import json
m = json.load(open('backend/model/metrics.json'))
print('extra datasets:', list(m['extra_datasets_sha256']))
print('rows after dedup:', m['audit']['retained_rows'], ' splits:', m['split_sizes'])
print('thresholds:', m['threshold_selection']['thresholds'])
t = m['test']
print('TEST raw SVM @0   P=%.3f R=%.3f F1=%.3f' % (t['raw_svm_default_zero']['precision'], t['raw_svm_default_zero']['recall'], t['raw_svm_default_zero']['f1']))
for lvl, d in t['sensitivities'].items():
    p = d['policy_toxicity']
    print('TEST %-8s policy P=%.3f R=%.3f F1=%.3f  safe-text FP rate=%.1f%%' % (lvl, p['precision'], p['recall'], p['f1'], 100*p['safe_text_false_positive_rate']))

## Step 5: download the trained model

Unzip the download into your project so the files land in `backend/model/`, then restart `python backend/app.py`. Local scikit-learn must match the version printed below (`pip install scikit-learn==<version>`), otherwise retrain locally.

In [ ]:
import shutil, sklearn
print('trained with scikit-learn', sklearn.__version__)
shutil.make_archive('/content/model_files', 'zip', 'backend/model')
files.download('/content/model_files.zip')